In [1]:
# Cell 1: Complete Pipeline (Run this cell)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor

# 1. Load and Clean Data
df = pd.read_csv('data/Uber-Jan-Feb-FOIL.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.day_name()
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)
df.dropna(inplace=True)

# 2. Machine Learning Features
df_ml = df.copy()
df_ml['base_code'] = df_ml['dispatching_base_number'].astype('category').cat.codes
df_ml['day_code'] = df_ml['day_of_week'].astype('category').cat.codes

X = df_ml[['active_vehicles', 'base_code', 'day_code', 'is_weekend']]
y = df_ml['trips']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train Models (XGBoost & Ensemble)
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gbr_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
ensemble_model = VotingRegressor(estimators=[('xgb', xgb_model), ('gbr', gbr_model)])

xgb_model.fit(X_train, y_train)
ensemble_model.fit(X_train, y_train)

xgb_mape = mean_absolute_percentage_error(y_test, xgb_model.predict(X_test)) * 100
ensemble_mape = mean_absolute_percentage_error(y_test, ensemble_model.predict(X_test)) * 100

print(f"XGBoost MAPE: {xgb_mape:.2f}%")
print(f"Ensemble MAPE: {ensemble_mape:.2f}%")

XGBoost MAPE: 8.90%
Ensemble MAPE: 8.14%
